# DATA LOADING PIPELINE (POLARS) - V1

This notebook contains **data loading and preprocessing pipeline** using **Polars** for optimal performance

**Purpose:**
- Load and validate datasets
- Apply discovered preprocessing strategies (WITHOUT winsorizing)
- Prepare data for modeling

**Datasets:**
- train_full: 5,337,414 rows, 94 columns
- test_full: 1,447,107 rows, 92 columns

**Key Preprocessing Steps:**
1. Load datasets with Polars
2. Apply missing value strategies (based on analysis)
3. Create missing flags and filled columns
4. Handle weight distribution issues
5. Export processed data for modeling

**Changes in V1:**
- Removed data winsorizing section (worsened prediction results)
- Save files with _v1 suffix

## 1.1 IMPORTS AND CONFIGURATION

Setting up environment with all necessary libraries and Polars configuration.

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter

# Polars configuration
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


## 1.2 DATA LOADING

Loading the full train and test datasets using Polars with performance timing.

In [2]:
# ============================================
# LOAD FULL DATASETS
# ============================================
print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

# Load full datasets using Polars
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
test_full = pl.read_parquet(full_test_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Test full: {test_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

print(f"\n✅ Full datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Test full: (1447107, 92)
Load time: 3.72 seconds

✅ Full datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## 1.3 DATA SIZE CHECKING

Checking the size of variables to fit them to float32

In [3]:
# ============================================
# DATA SIZE BEFORE OPTIMIZATION
# ============================================

def get_size_mb(df):
    return df.estimated_size() / 1024 ** 2


train_size_before = get_size_mb(train_full)
test_size_before = get_size_mb(test_full)

print("=" * 60)
print("DATA SIZE BEFORE OPTIMIZATION")
print("=" * 60)
print(f"Train size BEFORE: {train_size_before:.2f} MB")
print(f"Test size BEFORE : {test_size_before:.2f} MB")

DATA SIZE BEFORE OPTIMIZATION
Train size BEFORE: 3940.17 MB
Test size BEFORE : 1045.30 MB


## 2.0 DATA TYPE OPTIMIZATION

Optimize data types to reduce memory usage and improve performance.

In [4]:
# ============================================
# OPTIMIZE DATA TYPES
# ============================================
print("="*60)
print("OPTIMIZING DATA TYPES")
print("="*60)

# Get feature columns
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
print(f"Feature columns: {len(feature_cols)}")

# Convert feature columns to Float32
print("\n🔧 Converting feature columns to Float32...")
train_full = train_full.with_columns([
    pl.col(feature_cols).cast(pl.Float32)
])
test_full = test_full.with_columns([
    pl.col(feature_cols).cast(pl.Float32)
])

# Convert categorical columns to Categorical type
cat_cols = ['code', 'sub_code', 'sub_category']
print("\n🔧 Converting categorical columns to Categorical...")
train_full = train_full.with_columns([
    pl.col(cat_cols).cast(pl.Categorical)
])
test_full = test_full.with_columns([
    pl.col(cat_cols).cast(pl.Categorical)
])

# Convert integer columns to smaller types
int_cols = ['horizon', 'ts_index']
print("\n🔧 Converting integer columns to Int16...")
train_full = train_full.with_columns([
    pl.col(int_cols).cast(pl.Int16)
])
test_full = test_full.with_columns([
    pl.col(int_cols).cast(pl.Int16)
])

# Check new sizes
train_size_after = get_size_mb(train_full)
test_size_after = get_size_mb(test_full)

print("\n📊 SIZE REDUCTION:")
print(f"Train: {train_size_before:.2f} MB → {train_size_after:.2f} MB ({(1-train_size_after/train_size_before)*100:.1f}% reduction)")
print(f"Test:  {test_size_before:.2f} MB → {test_size_after:.2f} MB ({(1-test_size_after/test_size_before)*100:.1f}% reduction)")

print("\n✅ Data type optimization complete!")

OPTIMIZING DATA TYPES
Feature columns: 86

🔧 Converting feature columns to Float32...

🔧 Converting categorical columns to Categorical...

🔧 Converting integer columns to Int16...

📊 SIZE REDUCTION:
Train: 3940.17 MB → 2128.07 MB (46.0% reduction)
Test:  1045.30 MB → 553.99 MB (47.0% reduction)

✅ Data type optimization complete!


## 3.0 MISSING VALUE STRATEGIES

Apply missing value handling strategies based on analysis results.

In [ ]:
# ============================================
# MISSING VALUE STRATEGIES
# ============================================
print("="*60)
print("APPLYING MISSING VALUE STRATEGIES")
print("="*60)

# Priority columns based on missing value analysis
all_priority = ['feature_at', 'feature_by', 'feature_ay', 'feature_cd', 'feature_ce', 'feature_cf', 
               'feature_al', 'feature_aw', 'feature_bz', 'feature_bi', 'feature_i', 'feature_k', 
               'feature_h', 'feature_j', 'feature_cg', 'feature_av', 'feature_au', 'feature_ax']

print(f"Priority columns for missing value handling: {len(all_priority)}")

# Create missing flags and fill values for train data
print("\n🔧 Creating missing flags and filled columns (train)...")
train_exprs = []
for col in all_priority:
    if col in train_full.columns:
        train_exprs.extend([
            pl.col(col).is_null().cast(pl.Float32).alias(f"{col}_missing"),
            pl.col(col).fill_null(0.0).alias(f"{col}_filled")
        ])

train_full = train_full.with_columns(train_exprs)

# Create missing flags and fill values for test data
print("🔧 Creating missing flags and filled columns (test)...")
test_exprs = []
for col in all_priority:
    if col in test_full.columns:
        test_exprs.extend([
            pl.col(col).is_null().cast(pl.Float32).alias(f"{col}_missing"),
            pl.col(col).fill_null(0.0).alias(f"{col}_filled")
        ])

test_full = test_full.with_columns(test_exprs)

print(f"\n✅ Missing value strategies applied!")
print(f"   Train shape: {train_full.shape}")
print(f"   Test shape: {test_full.shape}")

## 3.1 WEIGHT FEATURES

Create weight-related features to handle extreme values and zero weights.

In [ ]:
# ============================================
# WEIGHT FEATURES
# ============================================
print("="*60)
print("CREATING WEIGHT FEATURES")
print("="*60)

# Analyze weight distribution
weight_stats = train_full.select([
    pl.col('weight').eq(0).sum().alias('zero_weights'),
    pl.col('weight').gt(1e9).sum().alias('high_weights'),
    pl.col('weight').max().alias('max_weight'),
    pl.col('weight').mean().alias('mean_weight')
])

zeros = weight_stats[0, 'zero_weights']
high = weight_stats[0, 'high_weights']
max_w = weight_stats[0, 'max_weight']
mean_w = weight_stats[0, 'mean_weight']

print(f"   Weight zeros: {zeros:,} ({zeros/len(train_full)*100:.3f}%)")
print(f"   Weight > 1e9: {high:,} ({high/len(train_full)*100:.3f}%)")
print(f"   Max weight: {max_w:,.2f}")
print(f"   Mean weight: {mean_w:,.2f}")

print("\n🔧 Creating weight features...")

# Create weight-related features
train_full = train_full.with_columns([
    pl.col('weight').log10().alias('weight_log10'),
    pl.col('weight').eq(0).cast(pl.Float32).alias('weight_is_zero'),
    pl.col('weight').gt(1e9).cast(pl.Float32).alias('weight_is_high'),
    pl.when(pl.col('weight').eq(0))
     .then(pl.lit('zero'))
     .when(pl.col('weight').gt(1e9))
     .then(pl.lit('high'))
     .otherwise(pl.lit('normal'))
     .alias('weight_category')
])

print("✅ weight_log10: Log10 transformation")
print("✅ weight_is_zero: Zero weight flag")
print("✅ weight_is_high: High weight (>1e9) flag")
print("✅ weight_category: Weight categorization")

# Show weight categories
weight_cats = train_full['weight_category'].value_counts()
print(f"\n📊 Weight categories:")
for cat in weight_cats['weight_category']:
    count = weight_cats.filter(pl.col('weight_category') == cat)['count'].item()
    print(f"   {cat}: {count:,} rows ({count/len(train_full)*100:.3f}%)")

In [ ]:
# ============================================
# APPLY WEIGHT FEATURES TO TEST DATA
# ============================================
print("🔧 APPLYING WEIGHT FEATURES TO TEST DATA...")

# Apply same weight features to test data
test_full = test_full.with_columns([
    pl.col('weight').log10().alias('weight_log10'),
    pl.col('weight').eq(0).cast(pl.Float32).alias('weight_is_zero'),
    pl.col('weight').gt(1e9).cast(pl.Float32).alias('weight_is_high'),
    pl.when(pl.col('weight').eq(0))
     .then(pl.lit('zero'))
     .when(pl.col('weight').gt(1e9))
     .then(pl.lit('high'))
     .otherwise(pl.lit('normal'))
     .alias('weight_category')
])

print("✅ weight_log10: Log10 transformation")
print("✅ weight_is_zero: Zero weight flag")
print("✅ weight_is_high: High weight (>1e9) flag")
print("✅ weight_category: Weight categorization")

# Show test weight categories
test_weight_cats = test_full['weight_category'].value_counts()
print(f"\n📊 Test weight categories:")
for cat in test_weight_cats['weight_category']:
    count = test_weight_cats.filter(pl.col('weight_category') == cat)['count'].item()
    print(f"   {cat}: {count:,} rows ({count/len(test_full)*100:.3f}%)")

## 4.0 FINAL DATA VALIDATION

Validate the processed datasets and save for modeling.

In [5]:
# ============================================
# FINAL DATA VALIDATION
# ============================================
print("📊 FINAL DATASET SUMMARY")
print("="*60)

# Calculate added columns
added_cols = len(all_priority) * 2 + 5  # flags + filled + weight features

print("\nTRAIN DATASET:")
print(f"   Shape: ({train_full.shape[0]:,}, {train_full.shape[1]})")
print(f"   Original columns: 94")
print(f"   Added columns: {added_cols}")
print(f"   Missing flags: {len(all_priority)}")
print(f"   Filled columns: {len(all_priority)}")
print(f"   Weight features: 5")

print("\nTEST DATASET:")
print(f"   Shape: ({test_full.shape[0]:,}, {test_full.shape[1]})")
print(f"   Original columns: 92")
print(f"   Added columns: {added_cols}")
print(f"   Missing flags: {len(all_priority)}")
print(f"   Filled columns: {len(all_priority)}")
print(f"   Weight features: 5")

print("\n✅ Data processing complete!")
print("   Ready for modeling with LightGBM/XGBoost")

📊 FINAL DATASET SUMMARY


NameError: name 'all_priority' is not defined

## 5.0 SAVE PROCESSED DATA

Save the processed datasets for modeling.

In [6]:
# ============================================
# SAVE PROCESSED DATA
# ============================================
print("💾 SAVING PROCESSED DATA...")

# Create processed data directory
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

# Save processed datasets with v1 suffix
train_path = processed_dir / "train_processed_v1.parquet"
test_path = processed_dir / "test_processed_v1.parquet"

print("⚡ Saving train_processed_v1.parquet...")
train_full.write_parquet(train_path)
print(f"✅ Train saved: {train_path}")

print("⚡ Saving test_processed_v1.parquet...")
test_full.write_parquet(test_path)
print(f"✅ Test saved: {test_path}")

print(f"\n📁 Files created:")
print(f"   train_processed_v1.parquet: {train_full.shape[0]:,} rows × {train_full.shape[1]} columns")
print(f"   test_processed_v1.parquet: {test_full.shape[0]:,} rows × {test_full.shape[1]} columns")

print("\n🎉 PIPELINE COMPLETE! Ready for modeling!")

💾 SAVING PROCESSED DATA...
⚡ Saving train_processed_v1.parquet...
✅ Train saved: ..\data\processed\train_processed_v1.parquet
⚡ Saving test_processed_v1.parquet...
✅ Test saved: ..\data\processed\test_processed_v1.parquet

📁 Files created:
   train_processed_v1.parquet: 5,337,414 rows × 94 columns
   test_processed_v1.parquet: 1,447,107 rows × 92 columns

🎉 PIPELINE COMPLETE! Ready for modeling!
